# 차세대 GUI 에이전트 파이프라인 실습

**💡 사전 필수 준비 (API Key 설정)**
1. Colab 왼쪽 사이드바에서 **'열쇠 아이콘'(보안 비밀)**을 클릭합니다.
2. 새 비밀 추가: 이름에 `GEMINI_API_KEY` 입력, 값에 API 키 붙여넣기
3. **'노트북 액세스' 토글 버튼을 켜서 활성화**합니다.

## 제1장. 개발 환경 구축 및 API 초기화

In [ ]:
!pip install -q -U google-genai pillow pydantic

import json
from google.colab import userdata
from google import genai
from google.genai import types
from IPython.display import Image, display
from PIL import Image as PILImage, ImageDraw, ImageFont
from pydantic import BaseModel, Field

# API 클라이언트 초기화
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("✅ Gemini API Client 초기화 성공!")
except userdata.SecretNotFoundError:
    print("❌ 에러: 'GEMINI_API_KEY'를 설정해주세요.")

# 라우팅을 위한 모델 정의
PLANNER_MODEL_ID = "gemini-1.5-pro" # 복잡한 추론 및 Task 분할용
EXECUTOR_MODEL_ID = "gemini-1.5-flash" # 빠르고 가벼운 좌표 추출 및 행동 수행용

## 제2장. 인지(Perception): 화면 추상화 및 Set-of-Mark (SoM)

In [ ]:
# 2.1 Colab 환경을 위한 가상의 애플리케이션 화면(로그인 폼) 생성 로직
def create_mock_gui_state():
    img = PILImage.new('RGB', (800, 600), color='#2b2b2b')
    draw = ImageDraw.Draw(img)

    # UI 요소 그리기
    draw.rectangle([200, 200, 600, 250], fill="#ffffff") # ID 입력창
    draw.rectangle([200, 300, 600, 350], fill="#ffffff") # PW 입력창
    draw.rectangle([300, 400, 500, 450], fill="#4285F4") # 로그인 버튼

    draw.text((210, 215), "Username", fill="#888888")
    draw.text((210, 315), "Password", fill="#888888")
    draw.text((375, 415), "LOGIN", fill="#ffffff")

    # Set-of-Mark (SoM) 시뮬레이션: 객체 탐지기가 마커를 달았다고 가정
    draw.ellipse([180, 215, 200, 235], fill="red")
    draw.text((185, 218), "1", fill="white") # ID 창 마커

    draw.ellipse([180, 315, 200, 335], fill="red")
    draw.text((185, 318), "2", fill="white") # PW 창 마커

    draw.ellipse([280, 415, 300, 435], fill="red")
    draw.text((285, 418), "3", fill="white") # 로그인 버튼 마커

    img.save("current_screen.png")
    return img

current_state_img = create_mock_gui_state()
print("✅ 현재 가상 화면 및 SoM 마커 생성 완료:")
display(current_state_img)

## 제3장 & 4장. 추론(Reasoning) 및 행동(Action): MCP Wrapping

In [ ]:
# 4.1 MCP Server Tool을 흉내내는 로컬 함수 (행동 모듈)
def mcp_tool_click(x: int, y: int):
    print(f"[MCP Execution] 🖱️ Mouse CLICKED at coordinates ({x}, {y})")

def mcp_tool_type(text: str):
    print(f"[MCP Execution] ⌨️ Keyboard TYPED: '{text}'")

# 3.1 구조화된 응답(JSON)을 강제하기 위한 Pydantic 스키마 설계
class GUIAction(BaseModel):
    thought: str = Field(description="행동을 결정한 논리적 이유")
    action_type: str = Field(description="'click' 또는 'type'")
    target_x: int = Field(description="클릭할 타겟의 x 중앙 좌표")
    target_y: int = Field(description="클릭할 타겟의 y 중앙 좌표")
    input_text: str = Field(description="type 액션일 경우 입력할 텍스트. click일 경우 빈 문자열")

# 3.2 Low-level Executor 에이전트 함수 (Spatial Grounding 담당)
def execute_gui_action(instruction: str, image_path: str):
    print(f"\n[Executor Agent] 🧠 주어진 지시문 분석 중: '{instruction}'")

    with open(image_path, "rb") as f:
        img_bytes = f.read()

    prompt = f"""You are a strict GUI execution agent.
    Task: {instruction}
    Analyze the screen image. The image has red circular markers with numbers (Set-of-Mark).
    Find the required element, estimate its center (x, y) coordinates. The image size is 800x600.
    Return EXACTLY the JSON format defined by the schema."""

    response = client.models.generate_content(
        model=EXECUTOR_MODEL_ID,
        contents=[
            types.Part.from_bytes(data=img_bytes, mime_type="image/png"),
            prompt
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=GUIAction,
            temperature=0.1 # 환각 억제를 위한 낮은 온도
        ),
    )

    action_data = json.loads(response.text)
    print(f"[Executor Agent] 🤔 판단 로직: {action_data['thought']}")

    # 4.2 의사결정에 따른 MCP Tool 호출 분기
    if action_data['action_type'] == 'click':
        mcp_tool_click(action_data['target_x'], action_data['target_y'])
    elif action_data['action_type'] == 'type':
        # 입력창 클릭 후 타이핑
        mcp_tool_click(action_data['target_x'], action_data['target_y'])
        mcp_tool_type(action_data['input_text'])
    else:
        print("알 수 없는 액션입니다.")

    return action_data

## 제5장. 고도화: Task Decomposition 기반 확장성 설계 (Scalable Reasoning)
전체 작업($N^2$ 복잡도)을 Planner가 작은 단위($N$)로 분할하여 Executor에게 넘기는 로직을 실증합니다.

In [ ]:
def scalable_gui_pipeline(user_goal: str):
    print("==================================================")
    print(f"🚀 [High-level Planner] 전체 목표 접수: {user_goal}")
    print("==================================================")

    # 실제 환경에서는 Gemini Pro가 이전 히스토리를 분석해 sub-task 배열을 생성합니다.
    # 이 실습에서는 복잡도 하향을 증명하기 위해 Planner의 Task Decomposition 결과를 하드코딩합니다.

    # [N^2 -> N 복잡도 하향 로직의 핵심]
    # 전체 히스토리를 매번 프롬프트에 넣지 않고, 분할된 원시 명령(Sub-task) 하나와
    # 현재 화면 1장만을 가벼운 Flash 모델(Executor)에 전달합니다.
    sub_tasks = [
        "Type the ID 'admin_user' into the input field indicated by marker 1.",
        "Type the password 'secret123' into the input field indicated by marker 2.",
        "Click the login button indicated by marker 3."
    ]

    for i, task in enumerate(sub_tasks):
        print(f"\n▶️ [Sub-Task {i+1} 실행 시작]")
        # 상태 추상화: 매 단계마다 현재 화면(캡처)를 업데이트하여 넘깁니다. (실습에서는 동일 이미지 사용)
        execute_gui_action(instruction=task, image_path="current_screen.png")
        time.sleep(2) # 다음 액션 전 UI 반응 대기 (시뮬레이션)

    print("\n✅ 모든 GUI Task가 완료되었습니다.")

# 전체 파이프라인 실행
scalable_gui_pipeline(user_goal="시스템에 관리자 계정(admin_user/secret123)으로 로그인 해줘.")